In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, bilinear, freqz, cheby1, dimpulse, dlti

def design_butterworth_filter(filter_order, cutoff_frequency, sampling_frequency):
    # Design the analog Butterworth filter
    analog_b, analog_a = butter(
        filter_order,
        cutoff_frequency,
        analog=True,
        btype='low'
    )
    # Perform the bilinear transformation
    digital_b, digital_a = bilinear(
        analog_b,
        analog_a,
        sampling_frequency
    )
    return digital_b, digital_a
def design_chebyshev_filter(
    filter_order,
    cutoff_frequency,
    sampling_frequency,
    ripple
):
    # Design the analog Chebyshev Type-I filter
    analog_b, analog_a = cheby1(
        filter_order,
        ripple,
        cutoff_frequency,
        analog=True,
        btype='low'
    )
    # Perform the bilinear transformation
    digital_b, digital_a = bilinear(
        analog_b,
        analog_a,
        sampling_frequency
    )
    return digital_b, digital_a
def plot_filter_response(digital_b, digital_a, sampling_frequency, filter_name):
    frequency, magnitude_response = freqz(
        digital_b,
        digital_a,
        fs=sampling_frequency,
        worN=4096
    )
    plt.figure(figsize=(10, 6))
    plt.plot(frequency, np.abs(magnitude_response))
    plt.title(filter_name + ' - Magnitude Response')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude')
    plt.grid(True)
    plt.show()
    system = dlti(digital_b, digital_a)
    time, impulse_response = dimpulse(
        system,
        n=50
    )
    impulse_response = np.squeeze(impulse_response)
    plt.figure(figsize=(10, 6))
    plt.stem(
        time,
        impulse_response,
        basefmt=" "
    )
    plt.title(filter_name + ' - Impulse Response')
    plt.xlabel('Samples')
    plt.ylabel('Amplitude')
    plt.grid(True)
    plt.show()
filter_order = 4
cutoff_frequency = 1000       # Hz
sampling_frequency = 8000     # Hz
ripple = 0.5                  # dB

digital_b, digital_a = design_butterworth_filter(
    filter_order,
    cutoff_frequency,
    sampling_frequency
)
plot_filter_response(
    digital_b,
    digital_a,
    sampling_frequency,
    'Butterworth Filter'
)
digital_b, digital_a = design_chebyshev_filter(
    filter_order,
    cutoff_frequency,
    sampling_frequency,
    ripple
)
plot_filter_response(
    digital_b,
    digital_a,
    sampling_frequency,
    'Chebyshev Type-I Filter'
)
filter_path = 'filter_coefficients.txt'
np.savetxt(
    filter_path,
    np.vstack((digital_b, digital_a)),
    delimiter=','
)
print(f"Filter coefficients saved at: {filter_path}")